In [ ]:
import kagglehub
path = kagglehub.dataset_download("senkin13/rdkit-2025-3-3-cp311")
print("Path to dataset files", path)
!pip install /kaggle/input/rdkit-2025-3-3-cp311/rdkit-2025.3.3-cp311-cp311-manylinux_2_28_x86_64.whl

In [ ]:
useless_cols = [   
    
    'MaxPartialCharge', 
    # Nan data
    'BCUT2D_MWHI',
    'BCUT2D_MWLOW',
    'BCUT2D_CHGHI',
    'BCUT2D_CHGLO',
    'BCUT2D_LOGPHI',
    'BCUT2D_LOGPLOW',
    'BCUT2D_MRHI',
    'BCUT2D_MRLOW',

    # Constant data
    'NumRadicalElectrons',
    'SMR_VSA8',
    'SlogP_VSA9',
    'fr_barbitur',
    'fr_benzodiazepine',
    'fr_dihydropyridine',
    'fr_epoxide',
    'fr_isothiocyan',
    'fr_lactam',
    'fr_nitroso',
    'fr_prisulfonamd',
    'fr_thiocyan',

    # High correlated data >0.95
    'MaxEStateIndex',
    'HeavyAtomMolWt',
    'ExactMolWt',
    'NumValenceElectrons',
    'Chi0',
    'Chi0n',
    'Chi0v',
    'Chi1',
    'Chi1n',
    'Chi1v',
    'Chi2n',
    'Kappa1',
    'LabuteASA',
    'HeavyAtomCount',
    'MolMR',
    'Chi3n',
    'BertzCT',
    'Chi2v',
    'Chi4n',
    'HallKierAlpha',
    'Chi3v',
    'Chi4v',
    'MinAbsPartialCharge',
    'MinPartialCharge',
    'MaxAbsPartialCharge',
    'FpDensityMorgan2',
    'FpDensityMorgan3',
    'Phi',
    'Kappa3',
    'fr_nitrile',
    'SlogP_VSA6',
    'NumAromaticCarbocycles',
    'NumAromaticRings',
    'fr_benzene',
    'VSA_EState6',
    'NOCount',
    'fr_C_O',
    'fr_C_O_noCOO',
    'NumHDonors',
    'fr_amide',
    'fr_Nhpyrrole',
    'fr_phenol',
    'fr_phenol_noOrthoHbond',
    'fr_COO2',
    'fr_halogen',
    'fr_diazo',
    'fr_nitro_arom',
    'fr_phos_ester'
]

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error


import networkx as nx
from rdkit.Chem import AllChem
from rdkit.Chem import Descriptors
from rdkit.Chem import rdmolops
from rdkit import Chem

import warnings
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

path = kagglehub.dataset_download("adamlogman/neurips-dataset")

print("Path to dataset files:", path)
X_train_dens = pd.read_csv('/kaggle/input/neurips-dataset/density.csv')
X_train_ffv = pd.read_csv('/kaggle/input/neurips-dataset/ffv.csv')
X_train_rg =  pd.read_csv('/kaggle/input/neurips-dataset/rg.csv')
X_train_tc = pd.read_csv('/kaggle/input/neurips-dataset/tc.csv')
X_train_tg =  pd.read_csv('/kaggle/input/neurips-dataset/tg.csv')

y_train_dens = X_train_dens['Density']
X_train_dens = X_train_dens.drop('Density',axis=1)
y_train_ffv = X_train_ffv['FFV']
X_train_ffv = X_train_ffv.drop('FFV',axis=1)
y_train_rg = X_train_rg['Rg']
X_train_rg = X_train_rg.drop('Rg',axis=1)
y_train_tc = X_train_tc['Tc']
X_train_tc = X_train_tc.drop('Tc',axis=1)
y_train_tg = X_train_tg['Tg']
X_train_tg = X_train_tg.drop('Tg',axis=1)

test =pd.read_csv('/kaggle/input/neurips-open-polymer-prediction-2025/test.csv')

In [ ]:
def make_smile_canonical(smile): 
    try:
        mol = Chem.MolFromSmiles(smile)
        canon_smile = Chem.MolToSmiles(mol, canonical=True)
        return canon_smile
    except:
        return np.nan
test['SMILES'] = test['SMILES'].apply(lambda s: make_smile_canonical(s))

In [ ]:
def preprocessing(df):
    desc_names = [desc[0] for desc in Descriptors.descList if desc[0] not in useless_cols]
    descriptors = [compute_all_descriptors(smi) for smi in df['SMILES'].to_list()]

    graph_feats = {'graph_diameter': [], 'avg_shortest_path': [], 'num_cycles': []}
    for smile in df['SMILES']:
         compute_graph_features(smile, graph_feats)
        
    result = pd.concat(
        [
            pd.DataFrame(descriptors, columns=desc_names),
            pd.DataFrame(graph_feats)
        ],
        axis=1
    )

    result = result.replace([-np.inf, np.inf], np.nan)
    return result



In [ ]:
def ensemble_predict(models, X):
    preds = np.array([model.predict(X) for model in models])
    return preds.mean(axis=0)

In [ ]:
def compute_all_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return [None] * len(desc_names)
    return [desc[1](mol) for desc in Descriptors.descList if desc[0] not in useless_cols]

def compute_graph_features(smiles, graph_feats):
    mol = Chem.MolFromSmiles(smiles)
    adj = rdmolops.GetAdjacencyMatrix(mol)
    G = nx.from_numpy_array(adj)

    graph_feats['graph_diameter'].append(nx.diameter(G) if nx.is_connected(G) else 0)
    graph_feats['avg_shortest_path'].append(nx.average_shortest_path_length(G) if nx.is_connected(G) else 0)
    graph_feats['num_cycles'].append(len(list(nx.cycle_basis(G))))

test = pd.concat([test, preprocessing(test)], axis=1)
test['Ipc']=np.log10(test['Ipc'])

test=test.drop(['id','SMILES'],axis=1)

In [ ]:
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge, Lasso
from sklearn.linear_model import HuberRegressor         
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.ensemble import HistGradientBoostingRegressor 



def predict(X_train,y_train,test,iterations,depth,leaf_reg,sample,est):
    models1 = []
        
    for i in range(50):
        model = CatBoostRegressor(
        iterations = iterations,
        max_depth=depth,
        l2_leaf_reg = leaf_reg,
        subsample = sample,
        random_state = i,
        verbose=0,  
        allow_writing_files=False
        )
        
        model.fit(X_train, y_train)
        preds = model.predict(X_train)
        if i%10==0:
            print(f"MSE of {i} is: ", mean_squared_error(y_train, preds))
        models1.append(model)
    
    for i in range(50):
        model = ExtraTreesRegressor(
        n_estimators=est,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features='sqrt',
        bootstrap=True,
        n_jobs=-1,
        random_state = i
        )
        
        model.fit(X_train, y_train)
        preds = model.predict(X_train)
        if i%10==0:
            print(f"MSE of {i} is: ", mean_squared_error(y_train, preds))
        models1.append(model)
    
    predict = ensemble_predict(models1,test)
    return predict
    


In [ ]:
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

def _ensemble_predict(models, test_data):
    
    all_predictions = np.zeros(len(test_data))
    for model in models:
        all_predictions += model.predict(test_data)
    return all_predictions / len(models)

def predict_lgbm_ensemble(X_train, y_train, test, model_type, n_models=50):
    #found the parameters using Optuna
    all_hyperparams = {
        'Rg': {
            'n_estimators': 3589, 'learning_rate': 0.01214378520374001,
            'num_leaves': 189, 'max_depth': 5, 'min_child_samples': 9,
            'feature_fraction': 0.8880203281936008, 'bagging_fraction': 0.5882680143406503,
            'bagging_freq': 2, 'lambda_l1': 1.4390390776106815e-07,
            'lambda_l2': 0.0010085580673799095
        },
        'Tc': {
            'n_estimators': 1752, 'learning_rate': 0.03794958201107575,
            'num_leaves': 186, 'max_depth': 10, 'min_child_samples': 45,
            'feature_fraction': 0.7628807483778035, 'bagging_fraction': 0.5352197708689208,
            'bagging_freq': 4, 'lambda_l1': 9.796022448320082e-05,
            'lambda_l2': 0.002956516655829614
        },
        'Tg': {
            'n_estimators': 807, 'learning_rate': 0.005830398807033883,
            'num_leaves': 54, 'max_depth': 7, 'min_child_samples': 16,
            'feature_fraction': 0.7528053024591003, 'bagging_fraction': 0.7822895510693105,
            'bagging_freq': 7, 'lambda_l1': 8.773496806835213e-08,
            'lambda_l2': 0.0010774087184100666
        },
        'Density': {
            'n_estimators': 3249, 'learning_rate': 0.032775021822542806,
            'num_leaves': 248, 'max_depth': 6, 'min_child_samples': 12,
            'feature_fraction': 0.8513019109191978, 'bagging_fraction': 0.7037499675614258,
            'bagging_freq': 1, 'lambda_l1': 0.010925374209247815,
            'lambda_l2': 9.101516243166537e-06
        },
        'FFV': {
            'n_estimators': 691, 'learning_rate': 0.09991672859219022,
            'num_leaves': 285, 'max_depth': 7, 'min_child_samples': 8,
            'feature_fraction': 0.9332772309027536, 'bagging_fraction': 0.8423283635266596,
            'bagging_freq': 5, 'lambda_l1': 0.026349160806162876,
            'lambda_l2': 0.0026070015145434325
        }
    }

    
    if model_type not in all_hyperparams:
        raise ValueError(f"Invalid model_type '{model_type}'. Please choose from {list(all_hyperparams.keys())}")

   
    params = all_hyperparams[model_type]
    
    models = []
    
    print(f"--- Training {n_models} LightGBM models for type: {model_type} ---")
    
    for i in range(n_models):
        model_params = params.copy()
        model_params['random_state'] = i
        model_params['n_jobs'] = -1 
        model_params['verbosity'] = -1 
        
        model = lgb.LGBMRegressor(**model_params)
        
        model.fit(X_train, y_train)
        models.append(model)

    print(f"Total models trained: {len(models)}")
    
    # Predict using the ensemble
    final_predictions = _ensemble_predict(models, test)
    return final_predictions

In [ ]:
ypred1 = predict_lgbm_ensemble(X_train_dens,y_train_dens,test,'Density')
ypred2 = predict_lgbm_ensemble(X_train_ffv,y_train_ffv,test,'FFV')
ypred3 = predict_lgbm_ensemble(X_train_rg,y_train_rg,test,'Tg')
ypred4 = predict_lgbm_ensemble(X_train_tc,y_train_tc,test,'Tc')
ypred5 = predict_lgbm_ensemble(X_train_tg,y_train_tg,test,'Rg')

In [ ]:
#submission to kaggle competition part
Submission = pd.read_csv('/kaggle/input/neurips-open-polymer-prediction-2025/test.csv')
Submission = Submission.drop('SMILES',axis = 1)
Submission['Density'] = ypred1
Submission['FFV'] = ypred2
Submission['Rg'] = ypred3
Submission['Tc']= ypred4 
Submission['Tg'] = ypred5 
submission = Submission
submission.to_csv("submission.csv", index=False)